# JobPilot AI: BiLSTM + Attention job recommendation

**Data-science notebook only.** This notebook discovers your resume and job files, inspects their real schemas, trains a resume-category classifier from scratch, and builds transparent rule-based matching tools. It does not create a Django UI or call external APIs, pretrained language models, or sentence-transformer embeddings.

Run cells from top to bottom in Google Colab. No dataset or test result is included or fabricated. The model cannot be trained or evaluated until you provide `Resume.csv` and `job_postings.csv`. Category confidence is the model's softmax score, not a calibrated probability or guarantee.

The classifier learns category patterns from labeled resume text. Skills, education, experience, ATS criteria, and job ranking are explicit rules and heuristics; their accuracy depends on your curated vocabulary and data quality.

## 1. Reproducible Colab environment

The pinned versions below form a compatible, reproducible baseline. If Colab reports that an already-loaded package was upgraded, restart the runtime and continue from the top.

In [ ]:
%pip install -q tensorflow==2.18.0 pandas==2.2.3 numpy==1.26.4 scikit-learn==1.6.1 PyMuPDF==1.25.3 openpyxl==3.1.5 joblib==1.4.2

In [ ]:
import os
import re
import sys
import json
import html
import random
import hashlib
import platform
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn
import fitz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print({
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
})
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Configure storage and discover data

Put files under one data directory (including nested company folders), or mount Google Drive / upload files below. Discovery is deliberately limited to `DATA_ROOT`, not the whole Colab filesystem. Set the paths to match your Drive. CSVs and each Excel sheet are inspected independently; parse errors are reported and are not silently ignored.

In [ ]:
# Choose one: a Drive folder, a local folder, or Colab uploads.
IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    DATA_ROOT = Path("/content/drive/MyDrive/JobPilot_AI_Datasets")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/JobPilot_AI_Artifacts")
else:
    DATA_ROOT = Path(os.environ.get("JOBPILOT_DATA_ROOT", str(Path.cwd())))
    ARTIFACT_DIR = Path(os.environ.get("JOBPILOT_ARTIFACT_DIR", str(Path.cwd() / "saved_models")))

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

UPLOAD_DATASETS = False  # Change to True in Colab to upload files interactively.
if UPLOAD_DATASETS:
    if not IN_COLAB:
        raise RuntimeError("The upload option requires Google Colab; set DATA_ROOT to a local directory instead.")
    from google.colab import files
    uploaded = files.upload()
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for uploaded_name, uploaded_bytes in uploaded.items():
        destination = DATA_ROOT / Path(uploaded_name).name
        destination.write_bytes(uploaded_bytes)
        print("Uploaded:", destination)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"DATA_ROOT does not exist: {DATA_ROOT}. Mount Drive, upload files, or change DATA_ROOT."
    )
print("Searching only under:", DATA_ROOT.resolve())

In [ ]:
data_files = sorted(
    path for path in DATA_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in {".csv", ".xls", ".xlsx"}
)
if not data_files:
    raise FileNotFoundError(f"No CSV/XLS/XLSX files found recursively under {DATA_ROOT.resolve()}.")
print(f"Discovered {len(data_files)} tabular file(s):")
for path in data_files:
    print(" -", path.relative_to(DATA_ROOT))

# Each entry is a distinct table. Excel workbooks may contribute multiple sheets.
tables = []
load_errors = []
for path in data_files:
    try:
        if path.suffix.lower() == ".csv":
            frame = pd.read_csv(path, low_memory=False)
            tables.append({"path": path, "sheet": None, "df": frame})
        else:
            workbook = pd.ExcelFile(path, engine="openpyxl")
            print(f"Excel workbook {path.name}; sheets: {workbook.sheet_names}")
            for sheet_name in workbook.sheet_names:
                frame = pd.read_excel(workbook, sheet_name=sheet_name)
                tables.append({"path": path, "sheet": sheet_name, "df": frame})
    except (OSError, ValueError, UnicodeError, ImportError) as exc:
        message = f"{path}: {type(exc).__name__}: {exc}"
        load_errors.append(message)
        print("LOAD ERROR:", message)

if load_errors:
    print(f"{len(load_errors)} file(s) could not be loaded. Resolve these errors if any are required.")
if not tables:
    raise RuntimeError("No tabular data could be loaded. Review the file paths and reported load errors.")

schema_rows = []
for index, item in enumerate(tables):
    frame = item["df"]
    location = f"{item['path']}" + (f" [sheet={item['sheet']}]" if item["sheet"] else "")
    print(f"\\nTABLE {index}: {location}")
    print(f"rows={len(frame):,}; columns={len(frame.columns)}")
    print("Columns:", frame.columns.tolist())
    print(frame.head(3).to_string(index=False, max_cols=20))
    schema_rows.append({
        "table_id": index, "path": str(item["path"]), "sheet": item["sheet"],
        "rows": len(frame), "columns": frame.columns.tolist(),
    })
schema_report = pd.DataFrame(schema_rows)
display(schema_report)

# Likely job-related supplementary tables are suggestions only. Do not merge them blindly.
JOB_HEADER_HINTS = ("job", "title", "description", "skill", "employer", "company", "occupation")
for index, item in enumerate(tables):
    normalized_headers = [str(column).casefold() for column in item["df"].columns]
    hints = [hint for hint in JOB_HEADER_HINTS if any(hint in header for header in normalized_headers)]
    if hints:
        print(f"Possible job-related table {index}: {item['path']} "
              f"sheet={item['sheet']!r}; matching header hints={hints}. Inspect before inclusion.")

## 3. Select and validate the core datasets and columns

The notebook prints observed columns above before resolving any fields. Exact file names are preferred, and duplicate matches require an explicit path. Column overrides are optional: if left blank, the notebook selects only a unique match from transparent header-name patterns and prints that choice. Ambiguity or a missing required field stops execution with the observed schema; edit the configuration rather than guessing. Supplemental files are not automatically merged.

In [ ]:
RESUME_DATASET_PATH = None  # Optional explicit full path if multiple Resume.csv files exist.
JOB_DATASET_PATH = None     # Optional explicit full path if multiple job_postings.csv files exist.
RESUME_TEXT_COLUMN = None
RESUME_LABEL_COLUMN = None
JOB_TITLE_COLUMN = None
JOB_DESCRIPTION_COLUMN = None
JOB_SKILLS_COLUMN = None    # Optional; set after inspecting actual columns.
JOB_ID_COLUMN = None        # Optional but recommended when genuine relevance labels use job IDs.

def choose_table(filename, explicit_path=None):
    if explicit_path is not None:
        target = Path(explicit_path).resolve()
        matches = [item for item in tables if item["path"].resolve() == target]
    else:
        matches = [item for item in tables if item["path"].name.casefold() == filename.casefold()]
    if len(matches) != 1:
        found = [str(item["path"]) for item in matches]
        raise ValueError(
            f"Expected exactly one {filename} table but found {len(matches)}: {found}. "
            f"Set the corresponding *_DATASET_PATH to an exact path. Discovered files: "
            f"{[str(path) for path in data_files]}"
        )
    return matches[0]

def resolve_column(frame, override, patterns, description, required=True):
    columns = list(frame.columns)
    if override is not None:
        if override not in columns:
            raise ValueError(f"{description} override {override!r} is absent. Observed columns: {columns}")
        return override
    normalized = {column: re.sub(r"[^a-z0-9]+", "_", str(column).casefold()).strip("_")
                  for column in columns}
    ranked = []
    for priority, pattern in enumerate(patterns):
        matches = [column for column, name in normalized.items() if name == pattern]
        if not matches:
            matches = [column for column, name in normalized.items()
                       if pattern in name.split("_")]
        if matches:
            ranked.append((priority, matches))
    if ranked:
        best_priority = min(priority for priority, _ in ranked)
        candidates = sorted({column for priority, found in ranked if priority == best_priority
                             for column in found}, key=str)
        if len(candidates) == 1:
            print(f"Selected {description}: {candidates[0]!r} from observed headers.")
            return candidates[0]
        raise ValueError(
            f"Ambiguous {description} candidates {candidates}. Set its override explicitly. "
            f"Observed columns: {columns}"
        )
    if required:
        raise ValueError(
            f"Could not identify {description}. Set its override after inspecting these columns: {columns}"
        )
    print(f"No {description} column selected; observed columns: {columns}")
    return None

resume_table = choose_table("Resume.csv", RESUME_DATASET_PATH)
job_table = choose_table("job_postings.csv", JOB_DATASET_PATH)
resume_df = resume_table["df"].copy()
jobs_raw = job_table["df"].copy()
if resume_df.empty or jobs_raw.empty:
    raise ValueError("Resume.csv and job_postings.csv must both contain at least one data row.")

print("Resume.csv observed columns:", resume_df.columns.tolist())
print("job_postings.csv observed columns:", jobs_raw.columns.tolist())
resume_text_col = resolve_column(
    resume_df, RESUME_TEXT_COLUMN,
    ["resume_str", "resume_text", "resume", "text", "content", "description"],
    "resume text",
)
resume_label_col = resolve_column(
    resume_df, RESUME_LABEL_COLUMN,
    ["category", "resume_category", "label", "class", "job_category"],
    "resume category label",
)
job_title_col = resolve_column(
    jobs_raw, JOB_TITLE_COLUMN,
    ["job_title", "title", "position", "role"],
    "job title",
)
job_description_col = resolve_column(
    jobs_raw, JOB_DESCRIPTION_COLUMN,
    ["job_description", "description", "job_desc", "details", "requirements"],
    "job description",
)
job_skills_col = resolve_column(
    jobs_raw, JOB_SKILLS_COLUMN,
    ["skills", "required_skills", "skill_set", "technical_skills"],
    "job skills",
    required=False,
)
job_id_col = resolve_column(
    jobs_raw, JOB_ID_COLUMN,
    ["job_id", "posting_id", "vacancy_id", "requisition_id", "id"],
    "job identifier",
    required=False,
)
print("Resolved columns:", {
    "resume_text": resume_text_col, "resume_label": resume_label_col,
    "job_title": job_title_col, "job_description": job_description_col,
    "job_skills": job_skills_col, "job_id": job_id_col,
})

## 4. Clean text, inspect labels, and create leakage-safe splits

Cleaning strips HTML and normalizes whitespace while preserving punctuation. Duplicate resume text/category pairs are removed before splitting. The split is stratified and the tokenizer/label encoder are fit only on training data. Each class needs enough examples for stratification; small datasets fail explicitly rather than silently producing invalid splits.

In [ ]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]*>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\\t\\r\\n]+", " ", text)
    return re.sub(r"\\s{2,}", " ", text).strip()

resume_data = pd.DataFrame({
    "text": resume_df[resume_text_col].map(clean_text),
    "label": resume_df[resume_label_col].map(
        lambda value: "" if value is None or pd.isna(value) else str(value).strip()
    ),
})
before_rows = len(resume_data)
resume_data = resume_data[(resume_data["text"] != "") & (resume_data["label"] != "")].copy()
removed_missing = before_rows - len(resume_data)
resume_data = resume_data.drop_duplicates(subset=["text", "label"], keep="first").reset_index(drop=True)
removed_duplicates = before_rows - removed_missing - len(resume_data)
resume_data["text_hash"] = resume_data["text"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)

label_counts = resume_data["label"].value_counts().sort_index()
print(f"Rows: input={before_rows}, removed_missing={removed_missing}, "
      f"duplicate_pairs_removed={removed_duplicates}, usable={len(resume_data)}")
print("Class counts (before split):")
display(label_counts.rename_axis("category").to_frame("count"))
if len(resume_data) < 10 or label_counts.size < 2:
    raise ValueError("Need at least 10 usable labeled resumes and at least 2 categories to train/evaluate.")
if int(label_counts.min()) < 5:
    raise ValueError(
        "Every category needs at least 5 usable examples for stratified train/validation/test splits. "
        "Collect more labeled data or explicitly redesign the evaluation protocol."
    )
conflicting_text_labels = resume_data.groupby("text")["label"].nunique()
conflicting_text_labels = conflicting_text_labels[conflicting_text_labels > 1]
if not conflicting_text_labels.empty:
    raise ValueError(
        f"{len(conflicting_text_labels)} cleaned resume text(s) have conflicting category labels. "
        "Resolve these labels before splitting to prevent identical text leaking across partitions."
    )

train_val_df, test_df = train_test_split(
    resume_data, test_size=0.20, random_state=SEED, stratify=resume_data["label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.20, random_state=SEED, stratify=train_val_df["label"]
)
print(f"Split sizes: train={len(train_df)}, validation={len(val_df)}, test={len(test_df)}")
print("Train class counts:")
display(train_df["label"].value_counts().sort_index().rename_axis("category").to_frame("count"))

MAX_VOCAB = 30000
MAX_SEQUENCE_LENGTH = 1200
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>", lower=True, filters="")
tokenizer.fit_on_texts(train_df["text"].tolist())
label_encoder = LabelEncoder()
label_encoder.fit(train_df["label"].tolist())
unknown_eval_labels = (set(val_df["label"]) | set(test_df["label"])) - set(label_encoder.classes_)
if unknown_eval_labels:
    raise ValueError(f"Evaluation labels absent from train split: {sorted(unknown_eval_labels)}")
split_hashes = {
    "train": set(train_df["text_hash"]),
    "validation": set(val_df["text_hash"]),
    "test": set(test_df["text_hash"]),
}
if (split_hashes["train"] & split_hashes["validation"] or
        split_hashes["train"] & split_hashes["test"] or
        split_hashes["validation"] & split_hashes["test"]):
    raise RuntimeError("Duplicate resume text crossed a split boundary; evaluation would leak.")

def encode_texts(texts):
    sequences = tokenizer.texts_to_sequences(list(texts))
    return pad_sequences(
        sequences, maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post"
    )

X_train = encode_texts(train_df["text"])
X_val = encode_texts(val_df["text"])
X_test = encode_texts(test_df["text"])
y_train = label_encoder.transform(train_df["label"])
y_val = label_encoder.transform(val_df["label"])
y_test = label_encoder.transform(test_df["label"])
print("Vocabulary size:", min(len(tokenizer.word_index) + 1, MAX_VOCAB))
print("Padded tensor shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Text columns are split before tokenizer fitting; test data is held out from model selection.")

## 5. Train a from-scratch BiLSTM with trainable Attention

All embedding and recurrent weights start from random initialization. The custom attention layer learns a score for each sequence position and applies the sequence mask. The validation split drives early stopping; the test split is used only after training to report held-out results.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="JobPilot")
class TrainableAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.score_layer = tf.keras.layers.Dense(1, use_bias=True)

    def call(self, inputs, mask=None):
        scores = tf.squeeze(self.score_layer(inputs), axis=-1)
        if mask is not None:
            scores = tf.where(tf.cast(mask, tf.bool), scores, tf.cast(-1e9, scores.dtype))
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * tf.expand_dims(weights, axis=-1), axis=1)

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        return super().get_config()

vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB)
if vocab_size < 2:
    raise ValueError("Training text produced no usable vocabulary tokens.")
embedding_dim = 128
inputs = tf.keras.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32", name="token_ids")
x = tf.keras.layers.Embedding(
    input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name="trainable_embedding"
)(inputs)
x = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.2), name="bilstm"
)(x)
x = TrainableAttention(name="trainable_attention")(x)
x = tf.keras.layers.Dropout(0.35)(x)
outputs = tf.keras.layers.Dense(len(label_encoder.classes_), activation="softmax", name="category")(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="jobpilot_bilstm_attention")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

class_ids = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=class_ids, y=y_train)
class_weights = {int(class_id): float(weight) for class_id, weight in zip(class_ids, weights)}
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-6, verbose=1),
]
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=15, batch_size=32, class_weight=class_weights,
    callbacks=callbacks, verbose=1,
)

## 6. Held-out classifier evaluation

These are actual test-set measurements from the trained model. Macro metrics weight categories equally; weighted metrics account for support. A confusion matrix and per-category support make weak or underrepresented classes visible. No results are reported until the training/evaluation cell has completed.

In [ ]:
test_probabilities = model.predict(X_test, verbose=0)
y_pred = np.argmax(test_probabilities, axis=1)
class_names = label_encoder.classes_.tolist()
test_accuracy = float(accuracy_score(y_test, y_pred))
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0
)
print(f"Held-out test accuracy: {test_accuracy:.4f}")
print(f"Macro precision={macro_p:.4f}; recall={macro_r:.4f}; F1={macro_f1:.4f}")
print(f"Weighted precision={weighted_p:.4f}; recall={weighted_r:.4f}; F1={weighted_f1:.4f}")
print("\\nClass-wise report:")
print(classification_report(
    y_test, y_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0
))
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
display(pd.DataFrame(cm, index=class_names, columns=class_names).rename_axis("actual").rename_axis("predicted", axis=1))
print("Test support by class:", dict(zip(class_names, np.bincount(y_test, minlength=len(class_names)))))
print("Interpret small or imbalanced test supports cautiously; this is not a guarantee of future performance.")

## 7. Explicit-rule resume information extraction

The category is learned by the BiLSTM. Skill extraction is not learned: it uses whole-phrase matching against a user-curated vocabulary, with aliases. The notebook does not claim to reliably discover arbitrary skills from text. Add your own terms after reviewing the source data; unknown skills remain unknown rather than being guessed. Education and experience are keyword/regex heuristics, not verified facts.

In [ ]:
# Populate with terms appropriate to your jobs; include canonical skills and spelling aliases.
# Example shape only; add/remove values based on your actual domain:
USER_SKILL_ALIASES = {
    # "python": ["python3"],
    # "machine learning": ["ml"],
}

def normalize_phrase(value):
    value = clean_text(value).casefold()
    value = re.sub(r"[^a-z0-9+#.]+", " ", value)
    return re.sub(r"\\s+", " ", value).strip()

skill_patterns = {}
for canonical, aliases in USER_SKILL_ALIASES.items():
    for phrase in [canonical, *aliases]:
        normalized = normalize_phrase(phrase)
        if normalized:
            skill_patterns[normalized] = str(canonical).strip()
skill_vocabulary = sorted(set(skill_patterns.values()), key=str.casefold)
if not skill_vocabulary:
    print("WARNING: curated skill vocabulary is empty; rule-based skill extraction/matching will be empty. "
          "Populate USER_SKILL_ALIASES from reviewed domain knowledge.")

def find_skills(text):
    normalized_text = f" {normalize_phrase(text)} "
    found = set()
    for phrase, canonical in skill_patterns.items():
        if f" {phrase} " in normalized_text:
            found.add(canonical)
    return sorted(found, key=str.casefold)

EDUCATION_TERMS = (
    "bachelor", "master", "ph.d", "phd", "doctorate", "associate",
    "b.tech", "m.tech", "b.e.", "m.e.", "mba", "degree", "diploma",
)
EXPERIENCE_PATTERN = re.compile(
    r"\\b(\\d{1,2}(?:\\.\\d+)?)\\s*\\+?\\s*(?:years?|yrs?)\\s+"
    r"(?:of\\s+)?(?:professional\\s+)?experience\\b", re.IGNORECASE
)

def extract_information(text):
    cleaned = clean_text(text)
    lowered = cleaned.casefold()
    experience_matches = [float(value) for value in EXPERIENCE_PATTERN.findall(cleaned)]
    education = sorted({term for term in EDUCATION_TERMS if term in lowered})
    return {
        "skills": find_skills(cleaned),
        "education_evidence": education,
        "experience_years_mentioned": max(experience_matches) if experience_matches else None,
        "unknown_skills": [],
        "unknown_skills_note": "Not inferred; add reviewed terms to USER_SKILL_ALIASES.",
    }

def predict_category(text):
    cleaned = clean_text(text)
    if not cleaned:
        raise ValueError("Cannot classify empty resume text.")
    probabilities = model.predict(encode_texts([cleaned]), verbose=0)[0]
    index = int(np.argmax(probabilities))
    return {
        "category": str(label_encoder.inverse_transform([index])[0]),
        "confidence": float(probabilities[index]),
        "confidence_note": "Uncalibrated softmax score; not a probability guarantee.",
    }

## 8. Prepare job postings and explainable recommendation scores

Only the configured core job table is included automatically. Supplementary company files remain separate until inspected and explicitly mapped. Similarity and skill overlap are ranking signals (0–100), not probabilities, hiring outcomes, or ground truth. Missing metadata is preserved as unknown; missing job skills can be matched from descriptions only against the curated vocabulary.

In [ ]:
jobs = pd.DataFrame({
    "job_title": jobs_raw[job_title_col].map(clean_text),
    "job_description": jobs_raw[job_description_col].map(clean_text),
})
if job_id_col is not None:
    jobs["job_id"] = jobs_raw[job_id_col].map(
        lambda value: "" if value is None or pd.isna(value) else str(value).strip()
    )
else:
    jobs["job_id"] = ""
if job_skills_col is None:
    jobs["job_skills_text"] = ""
else:
    jobs["job_skills_text"] = jobs_raw[job_skills_col].map(clean_text)
jobs["job_skills"] = [
    sorted(set(find_skills(description) + find_skills(skill_text)), key=str.casefold)
    for description, skill_text in zip(jobs["job_description"], jobs["job_skills_text"])
]
jobs = jobs[(jobs["job_title"] != "") | (jobs["job_description"] != "")].copy()
jobs = jobs.drop_duplicates(
    subset=["job_title", "job_description", "job_skills_text"], keep="first"
).reset_index(drop=True)
jobs.loc[jobs["job_id"] == "", "job_id"] = [
    f"generated_job_{index}" for index in range(int((jobs["job_id"] == "").sum()))
]
if jobs["job_id"].duplicated().any():
    duplicates = jobs.loc[jobs["job_id"].duplicated(keep=False), "job_id"].unique().tolist()
    raise ValueError(
        f"Job identifiers are not unique after deduplication: {duplicates[:20]}. "
        "Configure a unique job ID column or resolve duplicate source records."
    )
if jobs.empty:
    raise ValueError("No usable job postings remain after removing blank rows.")
print(f"Usable unique job postings: {len(jobs)}")
print("Jobs with no vocabulary-matched skills:", int(jobs["job_skills"].map(len).eq(0).sum()))
display(jobs[["job_id", "job_title", "job_skills"]].head(10))

def token_set(text):
    return set(re.findall(r"(?u)\\b\\w+\\b", clean_text(text).casefold()))

def score_job_resume(resume_text, job):
    resume_skills = set(find_skills(resume_text))
    job_skills = set(job["job_skills"])
    matched = sorted(resume_skills & job_skills, key=str.casefold)
    missing = sorted(job_skills - resume_skills, key=str.casefold)
    if job_skills:
        skill_coverage = len(matched) / len(job_skills)
    else:
        skill_coverage = None
    resume_tokens = token_set(resume_text)
    job_tokens = token_set(f"{job['job_title']} {job['job_description']}")
    text_overlap = (
        len(resume_tokens & job_tokens) / len(job_tokens) if job_tokens else 0.0
    )
    # Skill coverage has weight 70%, text-token coverage 30% when known skills exist.
    # Without known job skills the score uses text overlap alone and flags the limitation.
    score = 100 * (0.70 * skill_coverage + 0.30 * text_overlap) if skill_coverage is not None else 100 * text_overlap
    return {
        "match_score": round(float(score), 2),
        "matching_skills": matched,
        "missing_skills": missing,
        "skill_coverage": None if skill_coverage is None else round(float(skill_coverage), 4),
        "text_token_coverage": round(float(text_overlap), 4),
        "score_note": "Heuristic ranking score, not a probability or ground-truth evaluation.",
        "no_known_job_skills": not bool(job_skills),
    }

def recommend_jobs_for_resume(resume_text, top_k=10):
    cleaned = clean_text(resume_text)
    if not cleaned:
        raise ValueError("Resume text is empty; recommendations cannot be computed.")
    results = []
    for _, job in jobs.iterrows():
        result = score_job_resume(cleaned, job)
        results.append({
            "job_id": job["job_id"], "job_title": job["job_title"],
            "job_description": job["job_description"],
            **result,
        })
    ranked = pd.DataFrame(results).sort_values(
        ["match_score", "job_id"], ascending=[False, True], kind="mergesort"
    )
    return ranked.head(max(0, int(top_k))).reset_index(drop=True)

def rank_resumes_for_job(job_description, resumes, top_k=10, job_title="Provided job"):
    description = clean_text(job_description)
    if not description:
        raise ValueError("Job description is empty; recruiter ranking cannot be computed.")
    job_skills = find_skills(description)
    job = {"job_title": clean_text(job_title), "job_description": description, "job_skills": job_skills}
    results = []
    for resume_id, resume_text in resumes.items():
        result = score_job_resume(resume_text, job)
        results.append({"resume_id": resume_id, **result})
    return pd.DataFrame(results).sort_values(
        ["match_score", "resume_id"], ascending=[False, True], kind="mergesort"
    ).head(max(0, int(top_k))).reset_index(drop=True)

# Example API shapes using actual dataset rows (not evaluation labels).
example_resume_text = test_df.iloc[0]["text"]
display(recommend_jobs_for_resume(example_resume_text, top_k=10))
display(rank_resumes_for_job(
    jobs.iloc[0]["job_description"],
    {str(index): text for index, text in enumerate(test_df["text"].head(100))},
    top_k=10,
    job_title=jobs.iloc[0]["job_title"],
))

## 9. ATS-style heuristic and improvement suggestions

The score is a transparent resume-completeness heuristic, not a commercial ATS score or guarantee. Criteria and weights are explicit. If there are no known job skills, that criterion is excluded and remaining available weights are renormalized; unavailable information is not silently treated as evidence of a mismatch.

In [ ]:
ATS_WEIGHTS = {
    "section_completeness": 30.0,
    "target_skill_coverage": 35.0,
    "experience_signal": 15.0,
    "education_signal": 10.0,
    "length_readability": 10.0,
}
SECTION_HINTS = {
    "summary": ("summary", "profile", "objective"),
    "experience": ("experience", "employment", "work history"),
    "education": ("education", "academic"),
    "skills": ("skills", "technical skills", "competencies"),
    "projects": ("projects", "portfolio"),
}

def ats_score(resume_text, target_job=None):
    cleaned = clean_text(resume_text)
    if not cleaned:
        raise ValueError("Cannot score an empty resume.")
    lowered = cleaned.casefold()
    words = re.findall(r"\\b[\\w+#.]+\\b", cleaned)
    available = {}
    components = {}
    detected_sections = [
        section for section, hints in SECTION_HINTS.items()
        if any(re.search(rf"(?im)^\\s*{re.escape(hint)}\\s*:?[ \\t]*$", cleaned) for hint in hints)
    ]
    available["section_completeness"] = min(len(detected_sections) / len(SECTION_HINTS), 1.0)

    if target_job is not None:
        target_skills = set(find_skills(target_job))
        if target_skills:
            available["target_skill_coverage"] = len(set(find_skills(cleaned)) & target_skills) / len(target_skills)

    experience_found = bool(EXPERIENCE_PATTERN.search(cleaned)) or any(
        term in lowered for term in ("experience", "employment", "work history")
    )
    available["experience_signal"] = float(experience_found)
    education_found = any(term in lowered for term in EDUCATION_TERMS)
    available["education_signal"] = float(education_found)
    available["length_readability"] = min(len(words) / 300, 1.0) if len(words) < 300 else (
        1.0 if len(words) <= 1500 else max(0.0, 1 - (len(words) - 1500) / 1500)
    )

    available_weight = sum(ATS_WEIGHTS[name] for name in available)
    raw_score = sum(ATS_WEIGHTS[name] * value for name, value in available.items())
    score = round(100 * raw_score / available_weight, 2) if available_weight else 0.0
    suggestions = []
    missing_sections = sorted(set(SECTION_HINTS) - set(detected_sections))
    if missing_sections:
        suggestions.append("Add clearly labeled sections where relevant: " + ", ".join(missing_sections) + ".")
    if target_job is not None:
        target_skills = set(find_skills(target_job))
        missing_skills = sorted(target_skills - set(find_skills(cleaned)), key=str.casefold)
        if missing_skills:
            suggestions.append("If accurate, add evidence for relevant missing skills: " + ", ".join(missing_skills) + ".")
        elif not target_skills:
            suggestions.append("Curate the skill vocabulary to score skills for this job description.")
    if not experience_found:
        suggestions.append("Describe relevant experience and quantify duration/impact where truthful.")
    if not education_found:
        suggestions.append("Add relevant education or training details if applicable.")
    if len(words) < 300:
        suggestions.append("Consider adding concise, relevant evidence to provide enough context.")
    elif len(words) > 1500:
        suggestions.append("Consider shortening repetitive or unrelated content.")
    return {
        "score": score,
        "criteria_weights": {name: ATS_WEIGHTS[name] for name in available},
        "criteria_scores_0_to_1": {name: round(float(value), 4) for name, value in available.items()},
        "detected_sections": detected_sections,
        "word_count": len(words),
        "suggestions": suggestions,
        "note": "Heuristic completeness indicator only; not a real ATS compatibility result.",
    }

print(ats_score(example_resume_text, jobs.iloc[0]["job_description"]))

## 10. Recommendation evaluation (only with real relevance labels)

Ranking metrics require genuine resume-job relevance or interaction labels, not category labels or generated matches. Configure an explicit relevance file and observed column only if you have validated judgments (e.g., recruiter-reviewed relevance). Otherwise the notebook reports that recommendation ground truth is unavailable and does not invent precision, recall, or NDCG values.

In [ ]:
RELEVANCE_DATA_PATH = None  # Full path to an inspected, genuine resume-job relevance table.
RELEVANCE_JOB_ID_COLUMN = None
RELEVANCE_LABEL_COLUMN = None
RELEVANCE_RESUME_TEXT_COLUMN = None  # Explicit text column used to match judgments to held-out resumes.
RELEVANCE_POSITIVE_VALUES = {"1", "true", "yes", "relevant", "positive"}
RELEVANCE_NEGATIVE_VALUES = {"0", "false", "no", "irrelevant", "negative"}

if RELEVANCE_DATA_PATH is None:
    print("Recommendation precision@k/recall@k/NDCG@k not computed: no genuine relevance labels configured.")
else:
    relevance_path = Path(RELEVANCE_DATA_PATH)
    if not relevance_path.is_file():
        raise FileNotFoundError(f"Configured relevance file not found: {relevance_path}")
    relevance_df = pd.read_csv(relevance_path) if relevance_path.suffix.lower() == ".csv" else pd.read_excel(relevance_path)
    print("Observed relevance columns:", relevance_df.columns.tolist())
    for selected in (RELEVANCE_JOB_ID_COLUMN, RELEVANCE_LABEL_COLUMN):
        if selected is None or selected not in relevance_df.columns:
            raise ValueError(
                f"Set explicit relevance job ID and label columns from observed headers: "
                f"{relevance_df.columns.tolist()}"
            )
    if RELEVANCE_RESUME_TEXT_COLUMN is not None and RELEVANCE_RESUME_TEXT_COLUMN not in relevance_df.columns:
        raise ValueError(f"Resume text column {RELEVANCE_RESUME_TEXT_COLUMN!r} is absent.")
    if job_id_col is None:
        raise ValueError(
            "Recommendation evaluation needs a real job identifier column in job_postings.csv. "
            "Set JOB_ID_COLUMN to an inspected, unique identifier before evaluating."
        )
    normalized_labels = relevance_df[RELEVANCE_LABEL_COLUMN].astype(str).str.casefold()
    positive_values = {str(value).casefold() for value in RELEVANCE_POSITIVE_VALUES}
    negative_values = {str(value).casefold() for value in RELEVANCE_NEGATIVE_VALUES}
    unexpected_values = set(normalized_labels) - positive_values - negative_values
    if unexpected_values:
        raise ValueError(
            f"Unconfigured relevance label values: {sorted(unexpected_values)}. "
            "Add reviewed positive/negative values explicitly."
        )
    positive = normalized_labels.isin(positive_values)
    relevance_df = relevance_df.assign(_positive=positive)
    # Require a test-resume mapping, preventing evaluations on the training resumes.
    if RELEVANCE_RESUME_TEXT_COLUMN is None:
        raise ValueError(
            "Set RELEVANCE_RESUME_TEXT_COLUMN so relevance rows can be matched to held-out test resumes."
        )
    duplicate_judgments = relevance_df.duplicated(
        subset=[RELEVANCE_RESUME_TEXT_COLUMN, RELEVANCE_JOB_ID_COLUMN], keep=False
    )
    if duplicate_judgments.any():
        raise ValueError(
            "Relevance data contains duplicate resume/job judgments; resolve conflicting or repeated "
            "pairs before calculating ranking metrics."
        )
    if len(jobs) < 10:
        raise ValueError("At least 10 unique jobs are required to report precision@10.")
    test_hashes = set(test_df["text_hash"])
    relevance_df["_resume_hash"] = relevance_df[RELEVANCE_RESUME_TEXT_COLUMN].map(
        lambda value: hashlib.sha256(clean_text(value).encode("utf-8")).hexdigest()
    )
    relevance_df = relevance_df[relevance_df["_resume_hash"].isin(test_hashes)].copy()
    if relevance_df.empty:
        raise ValueError("No relevance judgments map to held-out test resumes; no ranking metrics are valid.")

    known_job_ids = set(jobs["job_id"].astype(str))
    relevance_job_ids = set(relevance_df[RELEVANCE_JOB_ID_COLUMN].astype(str))
    if relevance_job_ids - known_job_ids:
        raise ValueError(
            "Some relevance job IDs do not match the configured job table IDs: "
            f"{sorted(relevance_job_ids - known_job_ids)[:20]}"
        )

    def ranking_metrics_for_test(k=10):
        values = []
        for resume_hash, judged in relevance_df.groupby("_resume_hash"):
            row = test_df.loc[test_df["text_hash"] == resume_hash]
            if row.empty:
                continue
            relevant_ids = set(judged.loc[judged["_positive"], RELEVANCE_JOB_ID_COLUMN].astype(str))
            judged_ids = set(judged[RELEVANCE_JOB_ID_COLUMN].astype(str))
            if len(judged_ids) < k:
                raise ValueError(
                    f"Resume {resume_hash[:12]} has only {len(judged_ids)} judged jobs; "
                    f"at least k={k} are required for precision@k."
                )
            ranked_all = recommend_jobs_for_resume(row.iloc[0]["text"], top_k=len(jobs))
            ranked = ranked_all[ranked_all["job_id"].astype(str).isin(judged_ids)].head(k)
            ranked_ids = [str(value) for value in ranked["job_id"]]
            hits = np.array([1 if job_id in relevant_ids else 0 for job_id in ranked_ids], dtype=float)
            precision = float(hits.sum() / k)
            recall = float(hits.sum() / len(relevant_ids)) if relevant_ids else 0.0
            discounts = 1 / np.log2(np.arange(2, len(hits) + 2))
            dcg = float(np.sum(hits * discounts))
            ideal_hits = np.array([1] * min(len(relevant_ids), k) + [0] * max(0, k - len(relevant_ids)))
            idcg = float(np.sum(ideal_hits * discounts))
            values.append((precision, recall, dcg / idcg if idcg else 0.0, len(judged_ids)))
        if not values:
            raise ValueError("No held-out resume groups had evaluable relevance judgments.")
        return pd.DataFrame(values, columns=["precision_at_k", "recall_at_k", "ndcg_at_k", "judged_job_count"])

    evaluation = ranking_metrics_for_test(k=10)
    print(
        "Recommendation metrics over mapped held-out resumes and their user-judged job candidates. "
        "Unjudged jobs are excluded from each ranking; relevant labels are user-provided:"
    )
    display(evaluation[["precision_at_k", "recall_at_k", "ndcg_at_k"]].mean().to_frame("mean"))
    print("Evaluated resume groups:", len(evaluation))

## 11. Save the model bundle for later integration

Artifacts include the native Keras model, tokenizer JSON, label classes, curated skill vocabulary/aliases, preprocessing configuration, and metadata. The metadata records real observed test metrics and data provenance; do not treat these files as a production deployment until reviewed. `TrainableAttention` is registered for model loading.

In [ ]:
import joblib

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "jobpilot_bilstm_attention.keras"
tokenizer_path = ARTIFACT_DIR / "tokenizer.json"
label_encoder_path = ARTIFACT_DIR / "label_encoder.json"
skills_path = ARTIFACT_DIR / "skill_vocabulary.json"
preprocessing_path = ARTIFACT_DIR / "preprocessing_config.json"
metadata_path = ARTIFACT_DIR / "metadata.json"

model.save(model_path)
tokenizer_path.write_text(tokenizer.to_json(), encoding="utf-8")
label_encoder_path.write_text(
    json.dumps({"classes": label_encoder.classes_.tolist()}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
skills_path.write_text(
    json.dumps({"skills": skill_vocabulary, "aliases": USER_SKILL_ALIASES}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
preprocessing_config = {
    "text_cleaning": "HTML unescape/tag stripping, Unicode NFKC, whitespace normalization",
    "lowercase_tokenizer": True,
    "max_vocabulary": MAX_VOCAB,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "padding": "post",
    "truncating": "post",
    "seed": SEED,
    "resume_text_column": str(resume_text_col),
    "resume_label_column": str(resume_label_col),
    "job_title_column": str(job_title_col),
    "job_description_column": str(job_description_col),
    "job_skills_column": None if job_skills_col is None else str(job_skills_col),
    "job_id_column": None if job_id_col is None else str(job_id_col),
    "attention_layer": "TrainableAttention",
}
preprocessing_path.write_text(
    json.dumps(preprocessing_config, ensure_ascii=False, indent=2), encoding="utf-8"
)
metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_sources": {
        "resume": str(resume_table["path"]),
        "jobs": str(job_table["path"]),
    },
    "row_counts": {
        "resume_input": int(before_rows), "resume_usable": int(len(resume_data)),
        "resume_train": int(len(train_df)), "resume_validation": int(len(val_df)),
        "resume_test": int(len(test_df)), "jobs_usable": int(len(jobs)),
    },
    "resume_class_counts": {str(key): int(value) for key, value in label_counts.items()},
    "split_seed": SEED,
    "versions": {
        "python": platform.python_version(), "tensorflow": tf.__version__,
        "pandas": pd.__version__, "numpy": np.__version__, "scikit_learn": sklearn.__version__,
    },
    "test_metrics": {
        "accuracy": test_accuracy,
        "macro_precision": float(macro_p), "macro_recall": float(macro_r), "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p), "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    },
    "warning": "Softmax confidence is uncalibrated; recommendations and ATS scores are heuristics.",
}
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved model bundle to:", ARTIFACT_DIR.resolve())

## 12. Test on a genuinely unseen uploaded PDF

Choose a resume that was not used for training, validation, or testing. Text is extracted locally with PyMuPDF. Scanned/image-only PDFs produce an explicit error; if needed, run local OCR with Tesseract after installing/configuring it yourself. This notebook does not send resume data to external services. The output avoids printing contact details.

In [ ]:
def _extract_pdf_text(pdf_path):
    path = Path(pdf_path)
    if not path.is_file() or path.suffix.casefold() != ".pdf":
        raise FileNotFoundError(f"Expected an existing PDF file; got: {path}")
    page_text = []
    with fitz.open(path) as document:
        for page in document:
            page_text.append(page.get_text("text"))
    text = clean_text("\\n".join(page_text))
    if not text:
        raise ValueError(
            "No selectable text was extracted. This may be a scanned PDF; use local OCR "
            "(for example Tesseract) and provide the resulting text/PDF."
        )
    return text

def infer_resume_pdf(pdf_path, top_k=10):
    if model is None or tokenizer is None or label_encoder is None:
        raise RuntimeError("Train/load the model and its tokenizer/label encoder before inference.")
    text = _extract_pdf_text(pdf_path)
    resume_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    if resume_hash in set(resume_data["text_hash"]):
        raise ValueError(
            "This PDF's extracted text exactly matches a resume used in a dataset split. "
            "Choose a genuinely unseen resume."
        )
    prediction = predict_category(text)
    information = extract_information(text)
    recommendation_table = recommend_jobs_for_resume(text, top_k=top_k)
    target_description = (
        recommendation_table.iloc[0]["job_description"] if not recommendation_table.empty else None
    )
    score = ats_score(text, target_description)
    safe_columns = [
        "job_id", "job_title", "match_score", "matching_skills",
        "missing_skills", "score_note",
    ]
    return {
        **prediction,
        "extracted_information": information,
        "ats_score": score["score"],
        "ats_criteria": score["criteria_scores_0_to_1"],
        "suggestions": score["suggestions"],
        "top_jobs": recommendation_table[safe_columns].to_dict(orient="records"),
    }

if IN_COLAB:
    from google.colab import files
    uploaded_pdf = files.upload()
    pdf_names = [name for name in uploaded_pdf if name.casefold().endswith(".pdf")]
    if len(pdf_names) != 1:
        raise ValueError(f"Upload exactly one PDF resume; received PDF files: {pdf_names}")
    pdf_path = DATA_ROOT / Path(pdf_names[0]).name
    pdf_path.write_bytes(uploaded_pdf[pdf_names[0]])
    unseen_result = infer_resume_pdf(pdf_path)
    print(json.dumps(unseen_result, ensure_ascii=False, indent=2))
else:
    print("Not running in Colab. Set pdf_path to a genuinely unseen local PDF, then call infer_resume_pdf(pdf_path).")

## 13. Verify saved artifacts

This final cell fails if any required file is missing and prints absolute paths for later integration.

In [ ]:
required_artifacts = [
    model_path, tokenizer_path, label_encoder_path, skills_path, preprocessing_path, metadata_path
]
missing_artifacts = [path for path in required_artifacts if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError(f"Required artifacts missing: {[str(path) for path in missing_artifacts]}")
print("All saved artifacts verified:")
for path in required_artifacts:
    print(path.resolve())